In [ ]:
#---- Execute esse bloco caso queira baixar os dados diretamente do site oficiak kaggle ----
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ivanmitriakhin/arxiv-titles-abstracts-and-tags")

print("Path to dataset files:", path)

#lista os arquivos na pasta para ver o que foi baixado
import os
print("Arquivos no diretório:", os.listdir(path))
file_path = os.path.join(path, 'arxiv_data.csv')

100%|██████████| 860M/860M [00:27<00:00, 33.1MB/s] 

Extracting files...


Path to dataset files: C:\Users\Gustavo\.cache\kagglehub\datasets\ivanmitriakhin\arxiv-titles-abstracts-and-tags\versions\5
Arquivos no diretório: ['arxiv_data.csv', 'arxiv_data_grouped.csv', 'test', 'train']


# Tratamento de Dados

In [ ]:
# ---- Se voce executou os blocos de codigo acima, entao ignore esse bloco
# ---- Se voce ja fez a instalação do csv de forma manual (sem a execuçao do bloco de codigo acima, execute esse bloco
file_path = 'arxiv_data.csv'

# Aqui começa o tratamento de dados

In [ ]:
import pandas as pd


# Carregando dados
#arquivo = 'arxiv_data.csv'
#file_path = os.path.join(path, 'arxiv_data.csv')

data = pd.read_csv(file_path)

data

#Obtemos os nomes das colunas

variaveis = data.columns.to_list()
print(variaveis)

# Guardando colunas de interesse


colunasIniciais = ['ids', 'titles', 'abstracts']
categorias_interesses = ['cs.AI', 'cs.AR', 'cs.CC', 'cs.CE', 'cs.CG', 'cs.CL', 'cs.CR', 'cs.CV', 'cs.CY', 'cs.DB', 'cs.DC', 'cs.DL', 'cs.DM', 'cs.DS', 'cs.ET', 'cs.FL', 'cs.GL', 'cs.GR', 'cs.GT', 'cs.HC', 'cs.IR', 'cs.IT', 'cs.LG', 'cs.LO', 'cs.MA', 'cs.MM', 'cs.MS', 'cs.NA', 'cs.NE', 'cs.NI', 'cs.OH', 'cs.OS', 'cs.PF', 'cs.PL', 'cs.RO', 'cs.SC', 'cs.SD', 'cs.SE', 'cs.SI', 'cs.SY']
colunasMantidas = colunasIniciais + categorias_interesses


#Excluindo colunas que não nos interessa

colunasExcluir = [colunas for colunas in variaveis if colunas not in colunasMantidas]

data_final = data.drop(columns=colunasExcluir)
data_final


#Pegando uma fração dos dados, de forma aleatoria


data_final = data_final.sample(frac = 0.1, random_state=42)
data_final

# Mantendo apenas colunas que possuem valor 1 ou mais

colunas_de_categoria = data_final.columns[3:]
colunas_de_categoria
mascara_para_manter = (data_final[colunas_de_categoria] == 1).any(axis=1)

data_final = data_final[mascara_para_manter]
data_final

nome_do_arquivo = 'dados_tratados.csv'

data_final.to_csv(nome_do_arquivo, index=False, encoding='utf-8')

['ids', 'titles', 'abstracts', 'astro-ph.CO', 'astro-ph.EP', 'astro-ph.GA', 'astro-ph.HE', 'astro-ph.IM', 'astro-ph.SR', 'cond-mat.dis-nn', 'cond-mat.mes-hall', 'cond-mat.mtrl-sci', 'cond-mat.other', 'cond-mat.quant-gas', 'cond-mat.soft', 'cond-mat.stat-mech', 'cond-mat.str-el', 'cond-mat.supr-con', 'cs.AI', 'cs.AR', 'cs.CC', 'cs.CE', 'cs.CG', 'cs.CL', 'cs.CR', 'cs.CV', 'cs.CY', 'cs.DB', 'cs.DC', 'cs.DL', 'cs.DM', 'cs.DS', 'cs.ET', 'cs.FL', 'cs.GL', 'cs.GR', 'cs.GT', 'cs.HC', 'cs.IR', 'cs.IT', 'cs.LG', 'cs.LO', 'cs.MA', 'cs.MM', 'cs.MS', 'cs.NA', 'cs.NE', 'cs.NI', 'cs.OH', 'cs.OS', 'cs.PF', 'cs.PL', 'cs.RO', 'cs.SC', 'cs.SD', 'cs.SE', 'cs.SI', 'cs.SY', 'econ.EM', 'econ.GN', 'econ.TH', 'eess.AS', 'eess.IV', 'eess.SP', 'eess.SY', 'gr-qc', 'hep-ex', 'hep-lat', 'hep-ph', 'hep-th', 'math-ph', 'math.AC', 'math.AG', 'math.AP', 'math.AT', 'math.CA', 'math.CO', 'math.CT', 'math.CV', 'math.DG', 'math.DS', 'math.FA', 'math.GM', 'math.GN', 'math.GR', 'math.GT', 'math.HO', 'math.IT', 'math.KT', 'ma

# Matriz de Similaridade

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
# --- Carregando seus dados ---
try:
    df_artigos = pd.read_csv('dados_tratados.csv')
    
    # Tratamento de possíveis valores nulos nos textos
    df_artigos['titles'] = df_artigos['titles'].fillna('')
    df_artigos['abstracts'] = df_artigos['abstracts'].fillna('')

    # --- PASSO 1: Preparar o Conteúdo ---
    # Vamos combinar o título e o resumo em um único campo de texto ("documento")
    # para que o TF-IDF possa analisá-los juntos.
    df_artigos['conteudo_completo'] = df_artigos['titles'] + ' ' + df_artigos['abstracts']
    print("Coluna 'conteudo_completo' criada com sucesso.")

    # --- PASSO 2: Vetorização com TF-IDF ---
    print("Iniciando a vetorização com TF-IDF...")
    
    # Inicializa o vetorizador. 
    # stop_words='english' remove palavras comuns em inglês (como 'the', 'a', 'is')
    tfidf_vectorizer = TfidfVectorizer(stop_words='english')

    # 'fit_transform' aprende o vocabulário e transforma o texto em uma matriz numérica
    tfidf_matrix = tfidf_vectorizer.fit_transform(df_artigos['conteudo_completo'])
    
    print("Matriz TF-IDF criada com sucesso!")
    print(f"Dimensões da matriz: {tfidf_matrix.shape}") # (nº de artigos, nº de palavras únicas)

    # --- PASSO 3: Cálculo da Similaridade de Cosseno ---
    print("\nCalculando a matriz de similaridade de cosseno...")
    
    cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
    
    print("Matriz de similaridade calculada com sucesso!")
    print(f"Dimensões da matriz de similaridade: {cosine_sim_matrix.shape}")

    # Agora a 'cosine_sim_matrix' está pronta para ser usada pela sua função de recomendação!
    # Por exemplo, cosine_sim_matrix[0][1] te dará a similaridade entre o primeiro e o segundo artigo.

    # --- SALVANDO A MATRIZ ---
    print("\nSalvando a matriz de similaridade em um arquivo...")

    # np.save('nome_do_arquivo.npy', sua_matriz)
    np.save('cosine_similarity_matrix.npy', cosine_sim_matrix)

    print("Matriz salva com sucesso como 'cosine_similarity_matrix.npy'")

except FileNotFoundError:
    print("Erro: O arquivo 'dados_tratados.csv' não foi encontrado.")

Coluna 'conteudo_completo' criada com sucesso.
Iniciando a vetorização com TF-IDF...
Matriz TF-IDF criada com sucesso!
Dimensões da matriz: (19008, 52319)

Calculando a matriz de similaridade de cosseno...
Matriz de similaridade calculada com sucesso!
Dimensões da matriz de similaridade: (19008, 19008)

Salvando a matriz de similaridade em um arquivo...
Matriz salva com sucesso como 'cosine_similarity_matrix.npy'


# Simular Avaliações

In [5]:

import pandas as pd
import numpy as np
import random



# CONFIGURAÇÕES DA SIMULAÇÃO

NUM_USUARIOS = 500
NUM_AVALIACOES = 7000
ARQUIVO_ARTIGOS = 'dados_tratados.csv'
ARQUIVO_SAIDA = 'avaliacoes_simuladas.csv'

# Carregar dados dos artigos


try:
    df_artigos = pd.read_csv(ARQUIVO_ARTIGOS)
except FileNotFoundError:
    print(f"Erro: Arquivo '{ARQUIVO_ARTIGOS}' não encontrado. Certifique-se de que ele está no diretório correto.")
    exit()


# Identifica quais as colunas de categoria (todas após 'abstracts')

colunas_de_categoria = df_artigos.columns[3:].tolist()


# Definir os "Arquétipos" de Interesse

arquétipos = {
    "Especialista em IA e Machine Learning": [
        'cs.AI', # Inteligência Artificial
        'cs.LG', # Aprendizado de Máquina
        'cs.NE', # Computação Neural e Evolutiva
        'cs.CL'  # Linguística Computacional
    ],
    "Engenheiro de Sistemas e Hardware": [
        'cs.AR', # Arquitetura de Computadores
        'cs.OS', # Sistemas Operacionais
        'cs.DC', # Computação Distribuída, Paralela e em Cluster
        'cs.SY'  # Sistemas e Controle
    ],
    "Desenvolvedor de Software e Segurança": [
        'cs.SE', # Engenharia de Software
        'cs.PL', # Linguagens de Programação
        'cs.CR', # Criptografia e Segurança
        'cs.DB'  # Banco de Dados
    ],
    "Pesquisador em Teoria da Computação": [
        'cs.CC', # Complexidade Computacional
        'cs.GT', # Ciência da Computação e Teoria dos Jogos
        'cs.LO', # Lógica na Ciência da Computação
        'cs.DS'  # Estruturas de Dados e Algoritmos
    ],
    "Especialista em Robótica e Visão": [
        'cs.RO', # Robótica
        'cs.CV', # Visão Computacional
        'cs.AI', # Inteligência Artificial
        'cs.SY'  # Sistemas e Controle
    ],
    "Cientista de Dados e Informação": [
        'cs.IR', # Recuperação da Informação
        'cs.DB', # Banco de Dados
        'cs.DM', # Mineração de Dados
        'cs.SI'  # Análise de Redes Sociais e Informação
    ],
    "Especialista em Interação e Gráficos": [
        'cs.HC', # Interação Humano-Computador
        'cs.CG', # Computação Gráfica
        'cs.MM', # Multimídia
        'cs.GR'  # Gráficos
    ]
}


# Filtra os arquétipos para usar apenas categorias que realmente existem no seu CSV


arquétipos_validos = {}
for nome, categorias in arquétipos.items():
    categorias_presentes = [cat for cat in categorias if cat in colunas_de_categoria]
    if categorias_presentes:
        arquétipos_validos[nome] = categorias_presentes

if not arquétipos_validos:
    print("Nenhuma das categorias nos arquétipos foi encontrada no seu CSV. Usando todas as categorias.")
    arquétipos_validos = {"Geral": colunas_de_categoria}


# Criar os Perfis dos Usuários


perfis_usuarios = {}
lista_arquétipos = list(arquétipos_validos.keys())

for user_id in range(1, NUM_USUARIOS + 1):
    # Sorteia um arquétipo para o usuário
    tipo_perfil = random.choice(lista_arquétipos)
    # Associa as categorias daquele arquétipo ao usuário
    perfis_usuarios[user_id] = arquétipos_validos[tipo_perfil]

print(f"{len(perfis_usuarios)} perfis de usuários criados com sucesso.")

# Gerar Avaliações com Base no Perfils


lista_avaliacoes = []

for _ in range(NUM_AVALIACOES):
    # Escolhe um usuário aleatório pra fazer avaliação
    usuario_id = random.choice(list(perfis_usuarios.keys()))
    categorias_preferidas = perfis_usuarios[usuario_id]
    
    # 85% de chance de avaliar um artigo de sua área de interesse
    if random.random() < 0.85:
        # AVALIAÇÃO POSITIVA (dentro do perfil)
        
        # Filtra os artigos que pertencem a PELO MENOS UMA das categorias preferidas do usuário
        mascara_preferidos = df_artigos[categorias_preferidas].any(axis=1)
        artigos_candidatos = df_artigos[mascara_preferidos]
        
        if not artigos_candidatos.empty:
            artigo_escolhido = artigos_candidatos.sample(1)
            # Atribui uma nota alta (4 ou 5)
            nota = random.choice([4, 5])
        else:
            # Se não houver artigos no perfil pega um aleatório (caso raro)
            artigo_escolhido = df_artigos.sample(1)
            nota = 3

    else:
        # AVALIAÇÃO DE EXPLORAÇÃO (fora do perfil)
        
        # Filtra artigos que NÃO pertencem a NENHUMA das categorias preferidas
        mascara_preferidos = df_artigos[categorias_preferidas].any(axis=1)
        artigos_candidatos = df_artigos[~mascara_preferidos]

        if not artigos_candidatos.empty:
            artigo_escolhido = artigos_candidatos.sample(1)
            # Atribui uma nota baixa ou neutra (1, 2 ou 3)
            nota = random.choice([1, 2, 3])
        else:
            # Se todos os artigos forem do perfil do usuário pega um aleatório
            artigo_escolhido = df_artigos.sample(1)
            nota = 3
    
    # Adiciona a avaliação gerada a lista
    artigo_id = artigo_escolhido['ids'].iloc[0]
    lista_avaliacoes.append({
        'user_id': usuario_id,
        'artigo_id': artigo_id,
        'rating': nota
    })

# Criar e Salvar o DataFrame Final

df_avaliacoes_finais = pd.DataFrame(lista_avaliacoes)



# Remove duplicatas caso o mesmo usuário avalie o mesmo artigo duas vezes


df_avaliacoes_finais = df_avaliacoes_finais.drop_duplicates(subset=['user_id', 'artigo_id'], keep='last')

df_avaliacoes_finais.to_csv(ARQUIVO_SAIDA, index=False)

print(f"\nArquivo '{ARQUIVO_SAIDA}' gerado com sucesso!")
print(f"Total de avaliações únicas geradas: {len(df_avaliacoes_finais)}")
print("\n--- Amostra das Avaliações Geradas ---")
print(df_avaliacoes_finais.head())




500 perfis de usuários criados com sucesso.

Arquivo 'avaliacoes_simuladas.csv' gerado com sucesso!
Total de avaliações únicas geradas: 6994

--- Amostra das Avaliações Geradas ---
   user_id   artigo_id  rating
0      181  2006.02338       5
1      276  1903.11672       5
2      304  2309.05811       5
3      278  1210.04810       5
4      430  2211.05163       5


# Teste simples para verificar o funcionamento do sistema

In [4]:
import pandas as pd
import numpy as np

# DataFrame de artigos
df_artigos = pd.read_csv('dados_tratados.csv') 

# DataFrame de avaliações
df_avaliacoes = pd.read_csv('avaliacoes_simuladas.csv')

# matriz de similaridade
cosine_sim_matrix = np.load('cosine_similarity_matrix.npy')

# Um mapeamento do ID do artigo para o índice da linha no DataFrame
# encontrar rapidamente a posição de um artigo na matriz
indices = pd.Series(df_artigos.index, index=df_artigos['ids']).drop_duplicates()


# FUNÇÃO PRINCIPAL DE RECOMENDAÇÃO

def obter_recomendacoes(user_id, top_n=10):
    """
    Gera uma lista de recomendações para um usuário específico
    baseado no conteúdo dos artigos que ele já avaliou positivamente
    """
    
    # Encontrar artigos que usuário avaliou
    avaliacoes_do_usuario = df_avaliacoes[df_avaliacoes['user_id'] == user_id]
    
    # Filtrar para pegar apenas artigos que usuário gostou (nota >= 4)
    artigos_que_gostou = avaliacoes_do_usuario[avaliacoes_do_usuario['rating'] >= 4]
    
    if artigos_que_gostou.empty:
        return ["Não há avaliações positivas suficientes para gerar recomendações."]

    # Para cada artigo que usuário gostou, acumular scores de similaridade
    scores_agregados = {}
    for index, row in artigos_que_gostou.iterrows():
        artigo_id = row['artigo_id']
        nota = row['rating']
        
        # Pega índice do artigo na matriz de similaridade
        idx_artigo = indices[artigo_id]
        
        # Pega a linha de scores de similaridade para este artigo
        # e pondera pela nota (um item nota 5 tem mais influência)
        vetor_similaridade = cosine_sim_matrix[idx_artigo] * nota

        # Soma os scores ponderados no dicionário agregado
        for idx_similar, score in enumerate(vetor_similaridade):
            scores_agregados[idx_similar] = scores_agregados.get(idx_similar, 0) + score

    # Remover artigos que usuário já avaliou da lista de candidatos
    indices_ja_avaliados = [indices[artigo_id] for artigo_id in avaliacoes_do_usuario['artigo_id']]
    for idx in indices_ja_avaliados:
        if idx in scores_agregados:
            del scores_agregados[idx]

    # Ordenar artigos restantes pelo score agregado e pegar os 'top_n' melhores
    indices_recomendados = sorted(scores_agregados, key=scores_agregados.get, reverse=True)[:top_n]
    
    # Retornar os títulos dos artigos recomendados
    return df_artigos.iloc[indices_recomendados]['titles'].tolist()


# TESTANDO A FUNÇÃO
print("Função 'obter_recomendacoes' definida. Vamos testá-la...")

# Pega o primeiro ID de usuário único do dataset de avaliações para teste
id_usuario_teste = df_avaliacoes['user_id'].unique()[0]

print(f"\\n--- TESTE PARA O USUÁRIO: {id_usuario_teste} ---")

# Mostra o que esse usuário gostou pra podermos validar recomendação
avaliacoes_positivas_teste = df_avaliacoes[(df_avaliacoes['user_id'] == id_usuario_teste) & (df_avaliacoes['rating'] >= 4)]
titulos_gostou = df_artigos[df_artigos['ids'].isin(avaliacoes_positivas_teste['artigo_id'])]['titles'].tolist()

print("\\nEste usuário gostou de:")
for titulo in titulos_gostou:
    print(f"- {titulo}")

# Chama função para gerar recomendações
recomendacoes = obter_recomendacoes(user_id=id_usuario_teste, top_n=7)

print("\\nRecomendações geradas pelo sistema:")
for i, rec_titulo in enumerate(recomendacoes):
    print(f"{i+1}. {rec_titulo}")



# Bibliotecas necessárias pra interface
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# WIDGETS DA INTERFACE

# caixa de texto pro nome do novo usuário
input_usuario = widgets.Text(
    placeholder='Digite seu nome aqui...',
    description='Novo Usuário:',
    disabled=False
)

# Botão para iniciar o processo
botao_cadastrar = widgets.Button(
    description='Criar Perfil',
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Clique para iniciar a avaliação de artigos e criar seu perfil.',
    icon='user-plus'
)

# Container para os widgets de avaliação que serão criados dinamicamente
box_perfil_avaliacoes = widgets.VBox([])

# Container para a saída final das recomendações
output_final = widgets.Output()


# LÓGICA DA INTERFACE

def iniciar_criacao_perfil(b):
    """
    Esta função é chamada quando o botão 'Criar Perfil' é clicado
    Ela monta a tela de avaliação de artigos
    """
    # Pega o nome do usuário digitado
    nome_usuario = input_usuario.value
    if not nome_usuario:
        print("Por favor, digite um nome de usuário antes de continuar.")
        return

    # Limpa a tela anterior
    clear_output(wait=True)
    
    # Seleciona 10 artigos aleatórios para o usuário avaliar
    artigos_para_avaliar = df_artigos.sample(10)
    
    # Cria os componentes da tela de avaliação
    label_perfil = widgets.Label("Para entendermos seu gosto, avalie os seguintes artigos de 1 a 5:")
    
    # Cria um slider para cada artigo a ser avaliado
    sliders = []
    for _, row in artigos_para_avaliar.iterrows():
        slider = widgets.IntSlider(
            min=1, max=10, value=3, # Começa com nota 3 (neutra)
            description=row['titles'], 
            style={'description_width': 'initial'} # Evita que o título seja cortado
        )
        # Anexa o ID do artigo ao widget para uso posterior
        slider.artigo_id = row['ids'] 
        sliders.append(slider)
        
    botao_recomendar = widgets.Button(description='Gerar Recomendações', button_style='info', icon='lightbulb')
    
    # Define o que acontece quando o botão 'Gerar Recomendações' é clicado
    def obter_recomendacoes_novo_usuario(b):
        # Limpa a tela de avaliação
        clear_output(wait=True)
        # Mostra a área de saída final
        display(output_final)
        
        with output_final:
            print(f"Olá, {nome_usuario}! Com base no seu perfil, aqui estão suas recomendações:")
            
            # Coleta as avaliações dos sliders
            novas_avaliacoes = [{'user_id': nome_usuario, 'artigo_id': s.artigo_id, 'rating': s.value} for s in sliders]
            novas_avaliacoes_df = pd.DataFrame(novas_avaliacoes)
            
            # Adiciona as novas avaliações ao DataFrame global de avaliações
            global df_avaliacoes
            df_avaliacoes = pd.concat([df_avaliacoes, novas_avaliacoes_df], ignore_index=True)
            
            # Chama a sua função principal para obter as recomendações
            recomendacoes = obter_recomendacoes(user_id=nome_usuario, top_n=5)
            
            # Exibe as recomendações de forma legível
            print("-" * 50)
            for i, titulo in enumerate(recomendacoes):
                print(f"{i+1}. {titulo}")
            print("-" * 50)

    botao_recomendar.on_click(obter_recomendacoes_novo_usuario)
    
    # Monta e exibe a tela de avaliação
    box_perfil_avaliacoes.children = [label_perfil] + sliders + [botao_recomendar]
    display(box_perfil_avaliacoes)

# Conecta a função ao clique do botão de cadastro inicial
botao_cadastrar.on_click(iniciar_criacao_perfil)

Função 'obter_recomendacoes' definida. Vamos testá-la...
\n--- TESTE PARA O USUÁRIO: 7 ---
\nEste usuário gostou de:
- A Survey on Deep Reinforcement Learning-based Approaches for Adaptation
  and Generalization
- Online augmentation of learned grasp sequence policies for more
  adaptable and data-efficient in-hand manipulation
- Revisiting Permutation Symmetry for Merging Models between Different
  Datasets
- MMRDN: Consistent Representation for Multi-View Manipulation
  Relationship Detection in Object-Stacked Scenes
- Characterizing Trust and Resilience in Distributed Consensus for
  Cyberphysical Systems
- Towards the Localisation of Lesions in Diabetic Retinopathy
- On Data-Driven Log-Optimal Portfolio: A Sliding Window Approach
- High-Definition Map Generation Technologies For Autonomous Driving
\nRecomendações geradas pelo sistema:
1. SuperFusion: Multilevel LiDAR-Camera Fusion for Long-Range HD Map
  Generation
2. Benchmarking Robustness of Deep Reinforcement Learning approache

# Definindo artigos populares, para recomendar caso a recomendação nao retorne nada

In [5]:
# Conta quantas avaliações positivas (>=4) cada artigo recebeu
popularidade = df_avaliacoes[df_avaliacoes['rating'] >= 4]['artigo_id'].value_counts()

# Converte para um DataFrame e junta com os títulos
df_populares = pd.DataFrame({'ids': popularidade.index, 'contagem_positiva': popularidade.values})
df_populares = pd.merge(df_populares, df_artigos[['ids', 'titles']], on='ids')
df_populares = df_populares.sort_values(by='contagem_positiva', ascending=False)

print("Artigos mais populares:")
display(df_populares.head())

Artigos mais populares:


,ids,contagem_positiva,titles
0,1808.10826,5,Upward Planar Morphs
1,1505.06280,4,Risk-Sensitive Mean-Field-Type Games with Lp-n...
2,2006.03921,4,Robust watermarking with double detector-discr...
3,2012.12501,4,Learned Indexes for a Google-scale Disk-based ...
4,1703.10858,4,Language Oriented Modularity: From Theory to P...


# Main

In [ ]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Carregando todos os componentes necessários 

# DataFrame de artigos
df_artigos = pd.read_csv('dados_tratados.csv') 

# DataFrame de avaliações
df_avaliacoes = pd.read_csv('avaliacoes_simuladas.csv')

# Matriz de similaridade
cosine_sim_matrix = np.load('cosine_similarity_matrix.npy')

# 4. Mapeamento do ID do artigo para o índice da linha no DataFrame
indices = pd.Series(df_artigos.index, index=df_artigos['ids']).drop_duplicates()

# Cálculo da Popularidade dos Artigos
print("Calculando a popularidade dos artigos...")
popularidade = df_avaliacoes[df_avaliacoes['rating'] >= 4]['artigo_id'].value_counts()
df_populares = pd.DataFrame({'ids': popularidade.index, 'contagem_positiva': popularidade.values})
df_populares = pd.merge(df_populares, df_artigos[['ids', 'titles']], on='ids')
df_populares = df_populares.sort_values(by='contagem_positiva', ascending=False)
print("Cálculo de popularidade concluído.")


# FUNÇÃO PRINCIPAL DE RECOMENDAÇÃO 
def obter_recomendacoes(user_id, top_n=10):
    avaliacoes_do_usuario = df_avaliacoes[df_avaliacoes['user_id'] == user_id]
    
    # Tenta pegar artigos com nota alta (>= 4)
    artigos_que_gostou = avaliacoes_do_usuario[avaliacoes_do_usuario['rating'] >= 4]
    
    # Se não houver artigos com nota alta, tenta usar os de nota média (3)
    if artigos_que_gostou.empty:
        print("\n>> ... ...")
        artigos_que_gostou = avaliacoes_do_usuario[avaliacoes_do_usuario['rating'] == 3]
        
        # Se ainda assim não houver nenhum, retorna lista vazia
        if artigos_que_gostou.empty:
            return []
    
    scores_agregados = {}
    for index, row in artigos_que_gostou.iterrows():
        artigo_id = row['artigo_id']
        nota = row['rating']
        
        if artigo_id not in indices:
            continue
            
        idx_artigo = indices[artigo_id]
        vetor_similaridade = cosine_sim_matrix[idx_artigo] * nota

        for idx_similar, score in enumerate(vetor_similaridade):
            scores_agregados[idx_similar] = scores_agregados.get(idx_similar, 0) + score

    indices_ja_avaliados = [indices[artigo_id] for artigo_id in avaliacoes_do_usuario['artigo_id'] if artigo_id in indices]
    for idx in indices_ja_avaliados:
        if idx in scores_agregados:
            del scores_agregados[idx]

    indices_recomendados = sorted(scores_agregados, key=scores_agregados.get, reverse=True)[:top_n]
    
    return df_artigos.iloc[indices_recomendados]['titles'].tolist()

# FUNÇÃO ROBUSTA
def obter_recomendacoes_robusta(user_id, top_n=10):
    """
    Tenta obter recomendações por conteúdo
    Se falhar retorna os artigos mais populares que o usuário ainda não viu.
    """
    recomendacoes = obter_recomendacoes(user_id, top_n)
    
    if not recomendacoes:
        #print("\n>> Não encontramos recomendações personalizadas para o seu perfil.")
        print(">> Sugerindo os artigos mais populares que você ainda não viu:")
        
        artigos_ja_vistos = df_avaliacoes[df_avaliacoes['user_id'] == user_id]['artigo_id'].tolist()
        
        recomendacoes_populares = df_populares[~df_populares['ids'].isin(artigos_ja_vistos)]
        
        return recomendacoes_populares.head(top_n)['titles'].tolist()
    
    print("\n>> Recomendações personalizadas geradas com sucesso:")
    return recomendacoes


# LÓGICA DA INTERFACE
import ipywidgets as widgets
from IPython.display import display, clear_output

input_usuario = widgets.Text(placeholder='Digite seu nome aqui...', description='Novo Usuário:')
botao_cadastrar = widgets.Button(description='Criar Perfil', button_style='success', tooltip='Clique para criar seu perfil.', icon='user-plus')
box_perfil_avaliacoes = widgets.VBox([])
output_final = widgets.Output()

def iniciar_criacao_perfil(b):
    nome_usuario = input_usuario.value
    if not nome_usuario:
        print("Por favor, digite um nome de usuário antes de continuar.")
        return

    clear_output(wait=True)
    
    artigos_para_avaliar = df_artigos.sample(7)
    
    label_perfil = widgets.Label("Para entendermos seu gosto, avalie os seguintes artigos de 1 a 5:")
    
    sliders = []
    for _, row in artigos_para_avaliar.iterrows():
        # slider para a nota
        slider = widgets.IntSlider(
            min=1, max=5, value=3,
            step=1,
            description=row['titles'], 
            style={'description_width': 'initial'}, # Garante que o título não seja cortado
            layout=widgets.Layout(width='95%') # Ocupa a maior parte do espaço
        )
        # Anexa o ID do artigo ao widget pra usar depois
        slider.artigo_id = row['ids'] 
        sliders.append(slider)
        
    botao_recomendar = widgets.Button(description='Gerar Recomendações', button_style='info', icon='lightbulb')
    
    def obter_recomendacoes_novo_usuario(b):
        clear_output(wait=True)
        display(output_final)
        
        with output_final:
            print(f"Olá, {nome_usuario}! Processando seu perfil...")
            
            novas_avaliacoes = [{'user_id': nome_usuario, 'artigo_id': s.artigo_id, 'rating': s.value} for s in sliders]
            novas_avaliacoes_df = pd.DataFrame(novas_avaliacoes)
            
            global df_avaliacoes
            df_avaliacoes = pd.concat([df_avaliacoes, novas_avaliacoes_df], ignore_index=True)
            
            recomendacoes = obter_recomendacoes_robusta(user_id=nome_usuario, top_n=5)
            
            print("-" * 50)
            for i, titulo in enumerate(recomendacoes):
                print(f"{i+1}. {titulo}")
            print("-" * 50)

    botao_recomendar.on_click(obter_recomendacoes_novo_usuario)
    
    box_perfil_avaliacoes.children = [label_perfil] + sliders + [botao_recomendar]
    display(box_perfil_avaliacoes)

botao_cadastrar.on_click(iniciar_criacao_perfil)


# EXIBIÇÃO INICIAL DA INTERFACE
print("\nSistema de recomendação pronto.")
display(input_usuario, botao_cadastrar)

Output()